# Reward Fine-Tuning (Rejection Sampling / RAFT)

Improved the SFT model by training it on its own **best self-generated tours**.

**Method:** for each instance, sampled N tours, kept the shortest *feasible* one (verifiable reward = tour length), and fine-tuned on those. The objective is to concentrate probability on high-reward outputs.

**Before running:**
1. T4 GPU runtime.
2. Upload `train.jsonl`, `serialize.py`, `tsp_env.py` into `/content/`.
3. The SFT adapter must already be saved to Drive.

## 1. Installing dependencies

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12,<0.13" "datasets>=3.0,<3.2" \
    peft accelerate bitsandbytes "protobuf>=5.26,<6" "fsspec<=2024.9.0"
!pip uninstall -q -y torchao

## 2. Reloading the SFT model and **merging to 16-bit**

Generating from a 4-bit model is slow (~13 s/tour) because weights are dequantized every forward pass. Loading in 16-bit and merging the adapter with `merge_and_unload()` cuts generation to ~4.5 s. This becomes essential because harvesting requires thousands of generations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/content/drive/MyDrive/llm-tsp/sft-adapter-1.5b"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16,
                                            device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()      # fold adapter into weights for fast inference
print("merged 16-bit model ready")

## 3. Generation + scoring helpers

In [ ]:
import json, numpy as np
from serialize import parse_tour, is_feasible, SYSTEM_PROMPT, make_prompt, make_answer
from tsp_env import tour_length

def build_messages(coords):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": make_prompt(np.array(coords))}]

def generate_for(coords):
    msgs = build_messages(coords)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=128, do_sample=True,
                         temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

print("helpers ready")

## 4. Harvesting best-of-N tours (the rejection step)

For each instance, sampled N tours and kept the shortest **feasible** one. Infeasible samples are discarded — this filtering is the "rejection" in rejection sampling.

In [ ]:
def best_tour(coords, n_samples=3):
    n = len(coords); best, best_len = None, float('inf')
    for _ in range(n_samples):
        tour = parse_tour(generate_for(coords), n)
        if is_feasible(tour, n):
            L = tour_length(np.array(coords), tour)
            if L < best_len:
                best, best_len = tour, L
    return best

train = [json.loads(l) for l in open('/content/train.jsonl')]
subset = train[:300]          # subset keeps harvest time reasonable

raft_rows, kept = [], 0
for i, ex in enumerate(subset):
    tour = best_tour(ex['coords'], n_samples=3)
    if tour is not None:
        raft_rows.append({"messages": build_messages(ex['coords']) +
            [{"role": "assistant", "content": make_answer(tour)}]})
        kept += 1
    if (i + 1) % 25 == 0:
        print(f"  ...{i+1}/{len(subset)} processed, {kept} kept")

with open('/content/raft.jsonl', 'w') as f:
    for r in raft_rows:
        f.write(json.dumps(r) + '\n')
print(f"\nHarvested {kept} best tours from {len(subset)} instances")

## 5. Fine-tuning on the harvested tours

Added a fresh LoRA adapter on top of the merged SFT model and train on the self-generated best tours. Fewer epochs and a lower learning rate than the original SFT, since we are refining, not teaching from scratch.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

raft_data = load_dataset("json", data_files={"train": "/content/raft.jsonl"})

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"])

cfg = SFTConfig(
    output_dir="reward-qwen-tsp-1.5b",
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    gradient_checkpointing=True, num_train_epochs=2, learning_rate=1e-4,
    logging_steps=10, max_seq_length=2048, bf16=False, fp16=True, report_to="none")

trainer = SFTTrainer(model=model, args=cfg, train_dataset=raft_data["train"],
                     peft_config=lora, processing_class=tokenizer)
trainer.train()

## 6. Saved the reward adapter 

In [ ]:
save_path = '/content/drive/MyDrive/llm-tsp/reward-adapter-1.5b'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('saved to', save_path)